In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

MODEL_READY_DIR = PROJECT_ROOT / "data" / "processed" / "model_ready"

required_files = [
    "mat_X_train_balanced.csv",
    "mat_y_train_balanced.csv",
    "mat_X_test_processed.csv",
    "mat_y_test.csv",
    "por_X_train_balanced.csv",
    "por_y_train_balanced.csv",
    "por_X_test_processed.csv",
    "por_y_test.csv",
]

for filename in required_files:
    path = MODEL_READY_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

mat_X_train_balanced.csv: FOUND
mat_y_train_balanced.csv: FOUND
mat_X_test_processed.csv: FOUND
mat_y_test.csv: FOUND
por_X_train_balanced.csv: FOUND
por_y_train_balanced.csv: FOUND
por_X_test_processed.csv: FOUND
por_y_test.csv: FOUND


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load datasets
mat_X_train = pd.read_csv(MODEL_READY_DIR / "mat_X_train_balanced.csv")
mat_y_train = pd.read_csv(MODEL_READY_DIR / "mat_y_train_balanced.csv")

mat_X_test = pd.read_csv(MODEL_READY_DIR / "mat_X_test_processed.csv")
mat_y_test = pd.read_csv(MODEL_READY_DIR / "mat_y_test.csv")

por_X_train = pd.read_csv(MODEL_READY_DIR / "por_X_train_balanced.csv")
por_y_train = pd.read_csv(MODEL_READY_DIR / "por_y_train_balanced.csv")

por_X_test = pd.read_csv(MODEL_READY_DIR / "por_X_test_processed.csv")
por_y_test = pd.read_csv(MODEL_READY_DIR / "por_y_test.csv")

# Flatten target columns into Series
mat_y_train = mat_y_train.iloc[:, 0]
mat_y_test = mat_y_test.iloc[:, 0]

por_y_train = por_y_train.iloc[:, 0]
por_y_test = por_y_test.iloc[:, 0]

# Display dataset dimensions
datasets = {
    "Mathematics train": (mat_X_train, mat_y_train),
    "Mathematics test": (mat_X_test, mat_y_test),
    "Portuguese train": (por_X_train, por_y_train),
    "Portuguese test": (por_X_test, por_y_test),
}

for name, (X, y) in datasets.items():
    print(f"\n{name}")
    print("Features:", X.shape)
    print("Target:", y.shape)
    print("Target distribution:")
    print(y.value_counts().sort_index())

    assert len(X) == len(y), f"Row mismatch in {name}"
    assert not X.isnull().any().any(), f"Missing values in {name}"
    assert np.isfinite(X.select_dtypes(include=np.number)).all().all(), (
        f"Non-finite numeric values in {name}"
    )

print("\nBasic dataset validation passed.")


Mathematics train
Features: (424, 56)
Target: (424,)
Target distribution:
performance
0    212
1    212
Name: count, dtype: int64

Mathematics test
Features: (79, 56)
Target: (79,)
Target distribution:
performance
0    26
1    53
Name: count, dtype: int64

Portuguese train
Features: (878, 56)
Target: (878,)
Target distribution:
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test
Features: (130, 56)
Target: (130,)
Target distribution:
performance
0     20
1    110
Name: count, dtype: int64

Basic dataset validation passed.


In [3]:
# Verify that train and test feature columns match

for name, X_train, X_test in [
    ("Mathematics", mat_X_train, mat_X_test),
    ("Portuguese", por_X_train, por_X_test),
]:
    print(f"\n{name} dataset")

    print("Training feature count:", X_train.shape[1])
    print("Test feature count:", X_test.shape[1])

    train_cols = list(X_train.columns)
    test_cols = list(X_test.columns)

    missing_in_test = sorted(set(train_cols) - set(test_cols))
    extra_in_test = sorted(set(test_cols) - set(train_cols))

    print("Missing in test:", missing_in_test)
    print("Extra in test:", extra_in_test)

    assert train_cols == test_cols, (
        f"{name}: training and test feature columns differ"
    )

    assert X_train.select_dtypes(exclude="number").empty, (
        f"{name}: training data contains non-numeric features"
    )

    assert X_test.select_dtypes(exclude="number").empty, (
        f"{name}: test data contains non-numeric features"
    )

    print("Feature consistency verified.")

print("\nAll feature checks passed.")


Mathematics dataset
Training feature count: 56
Test feature count: 56
Missing in test: []
Extra in test: []
Feature consistency verified.

Portuguese dataset
Training feature count: 56
Test feature count: 56
Missing in test: []
Extra in test: []
Feature consistency verified.

All feature checks passed.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import numpy as np
import pandas as pd
import joblib

print("All model-training imports completed successfully.")

All model-training imports completed successfully.


In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5,
        n_jobs=-1
    ),

    "ANN": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=1000,
        early_stopping=True,
        random_state=42
    )
}

print("Models defined:", len(models))

for model_name in models:
    print("-", model_name)

Models defined: 6
- Logistic Regression
- Decision Tree
- Random Forest
- SVM
- KNN
- ANN


In [6]:

# Check data types of training features

print("MATHEMATICS TRAINING FEATURES")
print(mat_X_train.dtypes.value_counts())
print("\nNon-numeric columns:")
print(mat_X_train.select_dtypes(exclude="number").columns.tolist())

print("\nPORTUGUESE TRAINING FEATURES")
print(por_X_train.dtypes.value_counts())
print("\nNon-numeric columns:")
print(por_X_train.select_dtypes(exclude="number").columns.tolist())


MATHEMATICS TRAINING FEATURES
float64    56
Name: count, dtype: int64

Non-numeric columns:
[]

PORTUGUESE TRAINING FEATURES
float64    56
Name: count, dtype: int64

Non-numeric columns:
[]


In [7]:

# Check target labels and missing values

targets = {
    "Mathematics training target": mat_y_train,
    "Mathematics test target": mat_y_test,
    "Portuguese training target": por_y_train,
    "Portuguese test target": por_y_test,
}

for name, target in targets.items():
    values = set(target.dropna().unique())

    print(f"\n{name}")
    print("Unique labels:", sorted(values))
    print("Missing labels:", target.isna().sum())
    print("Label counts:")
    print(target.value_counts(dropna=False).sort_index())

    assert target.notna().all(), f"{name} contains missing labels!"
    assert values == {0, 1}, f"{name} does not contain both binary labels!"

print("\nAll target checks passed.")



Mathematics training target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    212
1    212
Name: count, dtype: int64

Mathematics test target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    26
1    53
Name: count, dtype: int64

Portuguese training target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test target
Unique labels: [np.int64(0), np.int64(1)]
Missing labels: 0
Label counts:
performance
0     20
1    110
Name: count, dtype: int64

All target checks passed.


In [8]:

# Compare training and test sizes

print("MATHEMATICS")
print("Training features:", mat_X_train.shape)
print("Training target:", mat_y_train.shape)
print("Test features:", mat_X_test.shape)
print("Test target:", mat_y_test.shape)

print("\nPORTUGUESE")
print("Training features:", por_X_train.shape)
print("Training target:", por_y_train.shape)
print("Test features:", por_X_test.shape)
print("Test target:", por_y_test.shape)

# Basic consistency checks
assert len(mat_X_train) == len(mat_y_train)
assert len(mat_X_test) == len(mat_y_test)
assert len(por_X_train) == len(por_y_train)
assert len(por_X_test) == len(por_y_test)

print("\nFeature/target row counts match.")


MATHEMATICS
Training features: (424, 56)
Training target: (424,)
Test features: (79, 56)
Test target: (79,)

PORTUGUESE
Training features: (878, 56)
Training target: (878,)
Test features: (130, 56)
Test target: (130,)

Feature/target row counts match.


In [9]:

print("MATHEMATICS NON-NUMERIC COLUMNS:")
print(mat_X_train.select_dtypes(exclude="number").columns.tolist())

print("\nPORTUGUESE NON-NUMERIC COLUMNS:")
print(por_X_train.select_dtypes(exclude="number").columns.tolist())


MATHEMATICS NON-NUMERIC COLUMNS:
[]

PORTUGUESE NON-NUMERIC COLUMNS:
[]


In [10]:

for name, target in {
    "Math train": mat_y_train,
    "Math test": mat_y_test,
    "Portuguese train": por_y_train,
    "Portuguese test": por_y_test,
}.items():
    print(f"\n{name}")
    print("Labels:", sorted(target.dropna().unique()))
    print("Missing values:", target.isna().sum())
    print(target.value_counts(dropna=False).sort_index())



Math train
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    212
1    212
Name: count, dtype: int64

Math test
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    26
1    53
Name: count, dtype: int64

Portuguese train
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0    439
1    439
Name: count, dtype: int64

Portuguese test
Labels: [np.int64(0), np.int64(1)]
Missing values: 0
performance
0     20
1    110
Name: count, dtype: int64


In [11]:

# Load the prepared CSV files for model training

mat_X_train = pd.read_csv(MODEL_READY_DIR / "mat_X_train_balanced.csv")
mat_y_train = pd.read_csv(MODEL_READY_DIR / "mat_y_train_balanced.csv").squeeze("columns")

mat_X_test = pd.read_csv(MODEL_READY_DIR / "mat_X_test_processed.csv")
mat_y_test = pd.read_csv(MODEL_READY_DIR / "mat_y_test.csv").squeeze("columns")

por_X_train = pd.read_csv(MODEL_READY_DIR / "por_X_train_balanced.csv")
por_y_train = pd.read_csv(MODEL_READY_DIR / "por_y_train_balanced.csv").squeeze("columns")

por_X_test = pd.read_csv(MODEL_READY_DIR / "por_X_test_processed.csv")
por_y_test = pd.read_csv(MODEL_READY_DIR / "por_y_test.csv").squeeze("columns")

print("All eight CSV files loaded successfully.")

print("\nMATHEMATICS")
print("Training features:", mat_X_train.shape)
print("Training target:", mat_y_train.shape)
print("Test features:", mat_X_test.shape)
print("Test target:", mat_y_test.shape)

print("\nPORTUGUESE")
print("Training features:", por_X_train.shape)
print("Training target:", por_y_train.shape)
print("Test features:", por_X_test.shape)
print("Test target:", por_y_test.shape)


All eight CSV files loaded successfully.

MATHEMATICS
Training features: (424, 56)
Training target: (424,)
Test features: (79, 56)
Test target: (79,)

PORTUGUESE
Training features: (878, 56)
Training target: (878,)
Test features: (130, 56)
Test target: (130,)
